# Comparacao sistematica de prompts: zero-shot vs few-shot vs CoT vs XML

Este notebook constroi um pequeno **harness de avaliacao**: um dataset com gabarito, varias variantes de prompt para a mesma tarefa, e um jeito objetivo de medir qual variante funciona melhor — em vez de julgar "no olho" olhando algumas respostas na interface de chat.

**Tarefa escolhida**: extrair `nome`, `data` e `valor` de um texto livre de recibo.

**Requer**: `pip install anthropic` e uma chave de API da Anthropic configurada como variavel de ambiente `ANTHROPIC_API_KEY` (ou substitua no codigo abaixo).

## 1. Dataset com gabarito

15 exemplos de recibos em texto livre, com a extracao correta anotada manualmente. Isso e o que torna a avaliacao objetiva: comparamos a saida do modelo contra um valor conhecido, campo a campo.

In [ ]:
dataset = [
    {"text": "Recibo: Joao Silva pagou R$ 150,00 em 12/03/2024 pelo servico de jardinagem.",
     "expected": {"nome": "Joao Silva", "data": "12/03/2024", "valor": "150,00"}},
    {"text": "Em 05/01/2024, Maria Oliveira efetuou pagamento de R$ 89,90 referente a consulta medica.",
     "expected": {"nome": "Maria Oliveira", "data": "05/01/2024", "valor": "89,90"}},
    {"text": "Pagamento recebido de Carlos Souza no valor de R$ 1.200,00, data: 20/07/2024.",
     "expected": {"nome": "Carlos Souza", "data": "20/07/2024", "valor": "1.200,00"}},
    {"text": "Ana Paula Ferreira quitou a mensalidade de R$ 320,50 no dia 01/02/2024.",
     "expected": {"nome": "Ana Paula Ferreira", "data": "01/02/2024", "valor": "320,50"}},
    {"text": "Nota de pagamento - Pedro Almeida - R$ 45,00 - 15/06/2024 - servico de encanamento.",
     "expected": {"nome": "Pedro Almeida", "data": "15/06/2024", "valor": "45,00"}},
    {"text": "Recebemos de Fernanda Costa a quantia de R$ 780,00 referente a aluguel, pago em 10/04/2024.",
     "expected": {"nome": "Fernanda Costa", "data": "10/04/2024", "valor": "780,00"}},
    {"text": "Comprovante: Lucas Martins, valor R$ 22,90, transacao realizada em 03/09/2024.",
     "expected": {"nome": "Lucas Martins", "data": "03/09/2024", "valor": "22,90"}},
    {"text": "Em 28/11/2024 Beatriz Lima pagou R$ 500,00 pela reforma do banheiro.",
     "expected": {"nome": "Beatriz Lima", "data": "28/11/2024", "valor": "500,00"}},
    {"text": "Recibo de prestacao de servico: Rafael Santos recebeu R$ 2.000,00 em 18/05/2024.",
     "expected": {"nome": "Rafael Santos", "data": "18/05/2024", "valor": "2.000,00"}},
    {"text": "Data: 07/08/2024. Cliente: Juliana Rocha. Valor pago: R$ 65,30.",
     "expected": {"nome": "Juliana Rocha", "data": "07/08/2024", "valor": "65,30"}},
    {"text": "Gustavo Pereira efetuou o pagamento de R$ 999,99 em 22/10/2024 pela mensalidade da academia.",
     "expected": {"nome": "Gustavo Pereira", "data": "22/10/2024", "valor": "999,99"}},
    {"text": "Recibo simplificado - nome: Camila Dias - total: R$ 310,00 - emitido 14/12/2024.",
     "expected": {"nome": "Camila Dias", "data": "14/12/2024", "valor": "310,00"}},
    {"text": "Pago por Thiago Nascimento o valor de R$ 48,00 referente a taxa de entrega, em 09/02/2024.",
     "expected": {"nome": "Thiago Nascimento", "data": "09/02/2024", "valor": "48,00"}},
    {"text": "Comprovante de deposito: Larissa Cardoso, R$ 1.500,00, 30/03/2024.",
     "expected": {"nome": "Larissa Cardoso", "data": "30/03/2024", "valor": "1.500,00"}},
    {"text": "Recebido de Bruno Teixeira: R$ 75,50, pago em 25/01/2024, referente a consultoria.",
     "expected": {"nome": "Bruno Teixeira", "data": "25/01/2024", "valor": "75,50"}},
]

print(f"Dataset com {len(dataset)} exemplos")


## 2. As variantes de prompt

Quatro abordagens diferentes para a mesma tarefa. Cada funcao recebe o texto do recibo e devolve o prompt completo a ser enviado ao modelo.

In [ ]:
def prompt_zero_shot(texto):
    return f"""Extraia o nome, a data e o valor do texto abaixo. Responda apenas com um JSON no formato {{"nome": "...", "data": "...", "valor": "..."}}.

Texto: {texto}"""


def prompt_few_shot(texto):
    exemplos = """Exemplo 1:
Texto: Marcos Vieira pagou R$ 300,00 em 01/01/2023 pelo aluguel.
Resposta: {"nome": "Marcos Vieira", "data": "01/01/2023", "valor": "300,00"}

Exemplo 2:
Texto: Em 15/05/2023, Patricia Gomes pagou R$ 45,00 pela consulta.
Resposta: {"nome": "Patricia Gomes", "data": "15/05/2023", "valor": "45,00"}

Exemplo 3:
Texto: Recibo de Andre Luiz, valor R$ 890,00, data 20/08/2023.
Resposta: {"nome": "Andre Luiz", "data": "20/08/2023", "valor": "890,00"}
"""
    return f"""Extraia o nome, a data e o valor do texto, seguindo o formato dos exemplos abaixo. Responda apenas com o JSON.

{exemplos}

Agora extraia deste texto:
Texto: {texto}
Resposta:"""


def prompt_cot(texto):
    return f"""Extraia o nome, a data e o valor do texto abaixo.

Antes de responder, pense passo a passo: identifique cada informacao no texto e onde ela aparece. Depois, na ultima linha, responda APENAS com o JSON no formato {{"nome": "...", "data": "...", "valor": "..."}}, sem nenhum texto depois dele.

Texto: {texto}"""


def prompt_xml(texto):
    return f"""Voce vai extrair informacoes estruturadas de um recibo.

<recibo>
{texto}
</recibo>

Extraia as seguintes informacoes do conteudo dentro da tag <recibo> e responda usando exatamente esta estrutura XML:

<resultado>
<nome>NOME_AQUI</nome>
<data>DATA_AQUI</data>
<valor>VALOR_AQUI</valor>
</resultado>"""


PROMPT_VARIANTS = {
    "zero-shot": prompt_zero_shot,
    "few-shot": prompt_few_shot,
    "chain-of-thought": prompt_cot,
    "xml-estruturado": prompt_xml,
}


## 3. Chamada ao modelo

Uma unica funcao que envia o prompt para a API da Anthropic e devolve o texto da resposta. Isolar essa chamada numa funcao facilita trocar de modelo ou de provedor depois sem mexer no resto do harness.

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def call_model(prompt, model="claude-sonnet-4-6", temperature=0.0):
    response = client.messages.create(
        model=model,
        max_tokens=300,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


## 4. Parsing e avaliacao

Duas etapas separadas, de propósito:

1. **Parsing**: extrair um dict `{nome, data, valor}` da resposta bruta do modelo, seja ela JSON ou XML
2. **Grading**: comparar o dict extraido contra o gabarito, campo a campo

Manter essas etapas separadas deixa claro onde um erro aconteceu: o modelo extraiu a informacao errada, ou o parsing falhou em interpretar uma resposta correta mas mal formatada?

In [ ]:
import re
import json

def parse_json_output(raw_text):
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def parse_xml_output(raw_text):
    result = {}
    for campo in ["nome", "data", "valor"]:
        match = re.search(rf"<{campo}>(.*?)</{campo}>", raw_text, re.DOTALL)
        if match:
            result[campo] = match.group(1).strip()
    return result if result else None

PARSERS = {
    "zero-shot": parse_json_output,
    "few-shot": parse_json_output,
    "chain-of-thought": parse_json_output,
    "xml-estruturado": parse_xml_output,
}

def grade(predicted, expected):
    if predicted is None:
        return {k: 0.0 for k in expected}
    scores = {}
    for k, v in expected.items():
        pred_val = str(predicted.get(k, "")).strip().lower()
        scores[k] = 1.0 if pred_val == v.strip().lower() else 0.0
    return scores


## 5. Rodar o harness completo

Para cada variante de prompt, rodamos contra todos os exemplos do dataset, fazemos o parsing e o grading, e guardamos os resultados. Isso pode levar alguns minutos dependendo do numero de exemplos e variantes (15 exemplos x 4 variantes = 60 chamadas de API).

In [ ]:
import pandas as pd
import time

results = []

for variant_name, prompt_fn in PROMPT_VARIANTS.items():
    parser = PARSERS[variant_name]
    for item in dataset:
        prompt = prompt_fn(item["text"])
        raw_output = call_model(prompt)
        predicted = parser(raw_output)
        scores = grade(predicted, item["expected"])

        results.append({
            "variante": variant_name,
            "texto": item["text"][:40] + "...",
            "nome_ok": scores["nome"],
            "data_ok": scores["data"],
            "valor_ok": scores["valor"],
            "media": sum(scores.values()) / len(scores),
        })
        time.sleep(0.5)  # evita rate limiting

df = pd.DataFrame(results)
df.head(10)


## 6. Agregar resultados por variante

In [ ]:
summary = df.groupby("variante")[["nome_ok", "data_ok", "valor_ok", "media"]].mean()
summary = summary.sort_values("media", ascending=False)
summary


In [ ]:
import matplotlib.pyplot as plt

summary["media"].plot(kind="bar", figsize=(7, 4), title="Acuracia media por variante de prompt")
plt.ylabel("Acuracia")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 7. Teste de consistencia (variancia entre execucoes)

Um prompt pode ter boa acuracia media mas ser inconsistente — respostas diferentes a cada execucao do mesmo input. Isso importa muito na pratica: um agente que muda de comportamento aleatoriamente é dificil de confiar.

Rodamos a melhor variante (segundo o resultado acima) 3 vezes contra os mesmos exemplos e medimos se a resposta muda.

In [ ]:
melhor_variante = summary.index[0]
prompt_fn = PROMPT_VARIANTS[melhor_variante]
parser = PARSERS[melhor_variante]

consistencia = []
for item in dataset[:5]:  # so os 5 primeiros, para nao gastar muitas chamadas
    respostas = []
    for _ in range(3):
        raw = call_model(prompt_fn(item["text"]), temperature=0.7)  # temperature > 0 para expor variacao
        pred = parser(raw)
        respostas.append(pred)
        time.sleep(0.5)

    # todas as 3 respostas sao identicas?
    identicas = all(r == respostas[0] for r in respostas)
    consistencia.append(identicas)
    print(f"{item['text'][:40]}... -> consistente: {identicas}")

taxa_consistencia = sum(consistencia) / len(consistencia)
print(f"\nTaxa de consistencia (temperature=0.7): {taxa_consistencia:.0%}")


## Notas finais

- **`temperature=0.0`** no harness principal reduz aleatoriedade — importante para comparar variantes de prompt de forma justa, sem que a diferença observada seja apenas ruído
- **Amostra pequena**: 15 exemplos é o suficiente para um primeiro sinal, mas para conclusões mais confiáveis vale expandir para 50-100 exemplos, especialmente se as variantes ficarem próximas em acurácia
- **Próximo passo natural**: este harness — chamar o modelo, fazer parsing, comparar contra gabarito, agregar métricas — é estruturalmente o mesmo tipo de código usado para avaliar **agentes** (Fase 5), só que mais simples porque aqui é uma única chamada, não um loop de múltiplos passos
- **Extensão**: adicione uma variante que combina few-shot + XML, ou teste o mesmo dataset em um modelo menor/mais barato para ver o trade-off custo vs. qualidade
